# 02 — End-to-End Training Workflow

**Project:** Diffusion LoRA Fine-Tuning on Product Images  
**Author:** Adebanji Oluwatimileyin Adelowo

This notebook covers the full training pipeline from config to checkpoint:
1. Load and validate config
2. Load base model components
3. Inject LoRA adapters
4. Build data loaders
5. Run training loop with logging
6. Save LoRA weights
7. Run validation inference

**Prerequisites:** Run `01_dataset_curation.ipynb` first to produce `data/metadata.csv`.

**Recommended:** Run on Kaggle (free T4) or Colab Pro (A100) — not CPU.

In [ ]:
# Verify GPU is available
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    raise RuntimeError("GPU required for training. Switch to a GPU runtime.")

print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
# !pip install diffusers transformers peft accelerate wandb bitsandbytes

import yaml
from pathlib import Path

CONFIG_PATH = "../configs/lora_config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print("Config loaded:")
print(f"  Base model:    {cfg['model']['base_model_id']}")
print(f"  LoRA rank:     {cfg['lora']['rank']}")
print(f"  Max steps:     {cfg['training']['max_train_steps']}")
print(f"  Resolution:    {cfg['dataset']['resolution']}")
print(f"  Output:        {cfg['training']['output_dir']}")

In [ ]:
## Step 1: Load base model components

from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer

model_id = cfg["model"]["base_model_id"]
dtype = torch.float16

print(f"Loading {model_id}...")

tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", torch_dtype=dtype)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

# Freeze non-LoRA components
vae.requires_grad_(False)
text_encoder.requires_grad_(False)

print("Base model loaded and frozen")

In [ ]:
## Step 2: Inject LoRA adapters into U-Net

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=cfg["lora"]["rank"],
    lora_alpha=cfg["lora"]["alpha"],
    target_modules=cfg["lora"]["target_modules"],
    lora_dropout=cfg["lora"]["dropout"],
    bias="none",
)

unet = get_peft_model(unet, lora_config)
unet.to(device)

unet.print_trainable_parameters()
# Expected output: ~0.3–0.8% of total parameters are trainable

In [ ]:
## Step 3: Build data loaders

import sys
sys.path.append("../scripts")
from dataset_loader import build_dataloaders

train_loader, val_loader = build_dataloaders(
    metadata_csv="../data/metadata.csv",
    tokenizer=tokenizer,
    resolution=cfg["dataset"]["resolution"],
    train_batch_size=cfg["training"]["train_batch_size"],
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader) if val_loader else 'N/A'}")

# Verify a batch
batch = next(iter(train_loader))
print(f"Batch pixel_values shape: {batch['pixel_values'].shape}")
print(f"Batch input_ids shape:    {batch['input_ids'].shape}")
print(f"Sample caption: {batch['caption'][0]}")

In [ ]:
## Step 4: Set up optimizer and LR scheduler

from diffusers.optimization import get_scheduler

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=cfg["training"]["learning_rate"],
)

max_steps = cfg["training"]["max_train_steps"]
grad_accum = cfg["training"]["gradient_accumulation_steps"]

lr_scheduler = get_scheduler(
    cfg["training"]["lr_scheduler"],
    optimizer=optimizer,
    num_warmup_steps=cfg["training"]["lr_warmup_steps"] * grad_accum,
    num_training_steps=max_steps * grad_accum,
)

print(f"Optimizer: AdamW lr={cfg['training']['learning_rate']}")
print(f"LR scheduler: {cfg['training']['lr_scheduler']}")
print(f"Max training steps: {max_steps}")

In [ ]:
## Step 5: Training loop
# For production runs, use scripts/train.py with accelerate launch.
# This notebook loop is for debugging and short test runs.

import wandb
from tqdm.auto import tqdm

# Optional: init W&B logging
# wandb.init(project="diffusion-lora-product", name=cfg["logging"]["run_name"])

global_step = 0
loss_history = []
unet.train()

progress = tqdm(total=max_steps, desc="Training")

for epoch in range(cfg["training"]["num_train_epochs"]):
    for step, batch in enumerate(train_loader):
        # Encode image to latent space
        with torch.no_grad():
            latents = vae.encode(
                batch["pixel_values"].to(device, dtype=dtype)
            ).latent_dist.sample() * vae.config.scaling_factor

        # Sample noise and add to latents
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps,
            (bsz,), device=device
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Encode text
        with torch.no_grad():
            encoder_hidden_states = text_encoder(batch["input_ids"].to(device))[0]

        # Predict noise
        model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = torch.nn.functional.mse_loss(model_pred.float(), noise.float())

        # Backward
        (loss / grad_accum).backward()
        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            loss_val = loss.item()
            loss_history.append((global_step, loss_val))
            progress.update(1)
            progress.set_postfix({"loss": f"{loss_val:.4f}", "step": global_step})

            # wandb.log({"loss": loss_val}, step=global_step)

        if global_step >= max_steps:
            break
    if global_step >= max_steps:
        break

progress.close()
print(f"Training complete. Final loss: {loss_history[-1][1]:.4f}")

In [ ]:
## Step 6: Plot training curve

import matplotlib.pyplot as plt

steps, losses = zip(*loss_history)
plt.figure(figsize=(10, 4))
plt.plot(steps, losses, alpha=0.4, label="raw")

# Smoothed
window = 20
smooth = [sum(losses[max(0,i-window):i+1]) / len(losses[max(0,i-window):i+1]) for i in range(len(losses))]
plt.plot(steps, smooth, label="smoothed", linewidth=2)

plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("LoRA Training Loss")
plt.legend()
plt.tight_layout()
plt.savefig("../results/training_loss.png", dpi=150)
plt.show()
print("Saved training_loss.png")

In [ ]:
## Step 7: Save LoRA weights

output_dir = Path(cfg["training"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

unet.save_pretrained(str(output_dir))

# Print checkpoint size
total_size = sum(f.stat().st_size for f in output_dir.rglob("*") if f.is_file())
print(f"LoRA checkpoint saved to {output_dir}")
print(f"Checkpoint size: {total_size / 1e6:.1f} MB")
print("(Compare: full SDXL = ~6.5 GB; LoRA adapter = ~50–100 MB)")
print("\nNEXT: Run 03_inference_comparison.ipynb")